In [7]:
from pyspark.sql import SparkSession

# Configurar Spark
spark = (SparkSession.builder
         .appName("LerHDFS")
         .master("local[*]")  # IP do Spark Master
         .config("spark.hadoop.fs.defaultFS", "hdfs://ha-cluster:9000")  # IP do NameNode
         .getOrCreate())

# Ler um arquivo no HDFS
df = spark.read.parquet("hdfs://ha-cluster:9000/landing/buyer/")

df.show()


25/01/10 21:44:45 WARN FileSystem: Failed to initialize fileystem hdfs://ha-cluster:9000/landing/buyer: java.lang.IllegalArgumentException: java.net.UnknownHostException: ha-cluster
25/01/10 21:44:45 WARN FileStreamSink: Assume no metadata directory. Error while looking for metadata directory in the path: hdfs://ha-cluster:9000/landing/buyer/.
java.lang.IllegalArgumentException: java.net.UnknownHostException: ha-cluster
	at org.apache.hadoop.security.SecurityUtil.buildTokenService(SecurityUtil.java:466)
	at org.apache.hadoop.hdfs.NameNodeProxiesClient.createProxyWithClientProtocol(NameNodeProxiesClient.java:134)
	at org.apache.hadoop.hdfs.DFSClient.<init>(DFSClient.java:374)
	at org.apache.hadoop.hdfs.DFSClient.<init>(DFSClient.java:308)
	at org.apache.hadoop.hdfs.DistributedFileSystem.initDFSClient(DistributedFileSystem.java:202)
	at org.apache.hadoop.hdfs.DistributedFileSystem.initialize(DistributedFileSystem.java:187)
	at org.apache.hadoop.fs.FileSystem.createFileSystem(FileSystem.j

IllegalArgumentException: java.net.UnknownHostException: ha-cluster

In [ ]:
from pyspark.sql import SparkSession

spark = (SparkSession.builder
         .appName("LerHDFS")
         .master("local[*]")
         .config("spark.hadoop.fs.defaultFS", "hdfs://localhost:9001")
         # Configurações adicionais que podem ajudar
         .config("spark.driver.host", "localhost")
         .config("spark.hadoop.dfs.client.use.datanode.hostname", "true")
         # Se estiver usando autenticação
         .config("spark.hadoop.user.name", "hdfs")  # ajuste o usuário se necessário
         .getOrCreate())

# Tente ler o arquivo
df = spark.read.parquet("hdfs://localhost:9001/landing/buyer/")

In [ ]:
spark = (SparkSession.builder
         .appName("LerHDFS")
         .master("local[*]")
         .config("spark.hadoop.fs.defaultFS", "hdfs://localhost:9001")
         .config("spark.driver.host", "localhost")
         .config("spark.hadoop.dfs.client.use.datanode.hostname", "true")
         # Adicione estas configurações
         .config("spark.hadoop.dfs.datanode.use.datanode.hostname", "true")
         .config("spark.hadoop.dfs.client.use.dns.nameserver", "true")
         .config("spark.hadoop.dfs.datanode.dns.interface", "default")
         .getOrCreate())

df = spark.read.parquet("hdfs://localhost:9001/landing/buyer/")
df.show()

In [8]:
from pyspark.sql import SparkSession
spark = (SparkSession.builder
         .appName("LerHDFS")
         .master("local[*]")
         # Usando o IP do secondnode (nó ativo)
         .config("spark.hadoop.fs.defaultFS", "hdfs://172.18.0.3:9000")
         .config("spark.driver.host", "localhost")
         .config("spark.hadoop.dfs.client.use.datanode.hostname", "false")  # mudamos para false pois usaremos IPs
         # Configurações adicionais para usar IPs
         .config("spark.hadoop.dfs.datanode.address", "172.18.0.4:9866")  # IP do datanode1
         .getOrCreate())

# Ler usando o mesmo endereço
df = spark.read.parquet("hdfs://172.18.0.3:9000/landing/buyer/")

25/01/10 21:44:54 WARN SparkSession: Using an existing Spark session; only runtime SQL configurations will take effect.
25/01/10 21:45:05 WARN BlockReaderFactory: I/O error constructing remote block reader.
org.apache.hadoop.net.ConnectTimeoutException: 60000 millis timeout while waiting for channel to be ready for connect. ch : java.nio.channels.SocketChannel[connection-pending remote=/172.18.0.4:9866]
	at org.apache.hadoop.net.NetUtils.connect(NetUtils.java:589)
	at org.apache.hadoop.hdfs.DFSClient.newConnectedPeer(DFSClient.java:3033)
	at org.apache.hadoop.hdfs.client.impl.BlockReaderFactory.nextTcpPeer(BlockReaderFactory.java:829)
	at org.apache.hadoop.hdfs.client.impl.BlockReaderFactory.getRemoteBlockReaderFromTcp(BlockReaderFactory.java:754)
	at org.apache.hadoop.hdfs.client.impl.BlockReaderFactory.build(BlockReaderFactory.java:381)
	at org.apache.hadoop.hdfs.DFSInputStream.getBlockReader(DFSInputStream.java:755)
	at org.apache.hadoop.hdfs.DFSInputStream.blockSeekTo(DFSInputStrea

In [ ]:
from pyspark.sql import SparkSession
spark = (SparkSession.builder
         .appName("LerHDFS")
         .master("local[*]")
         # Usando o endereço especial do Docker no WSL
         .config("spark.hadoop.fs.defaultFS", "hdfs://172.18.0.3:9000")
         .config("spark.driver.host", "host.docker.internal")
         .config("spark.hadoop.dfs.client.use.datanode.hostname", "false")
         # Adicionar resolução de hostname do Docker
         .config("spark.driver.extraJavaOptions", "-Dhost.docker.internal=host-gateway")
         .getOrCreate())

df = spark.read.parquet("hdfs://172.18.0.3:9000/landing/buyer/")

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, cast

# Inicializa Spark
spark = SparkSession.builder.appName("SelectExpr Example").getOrCreate()

# Exemplo de DataFrame
data = [("1", "2.0"), ("3", "4.5"), ("5", "6.7")]
columns = ["col1", "coln"]
df = spark.createDataFrame(data, columns)

# Usando selectExpr para fazer casts
# result_df = df.selectExpr("cast(col1 as int) as col1", "cast(coln as int) as coln")

# result_df.explain()


In [ ]:
df.select(
            col("col1").cast("int").alias("coluna_um"),
            col("coln").cast("float").alias("coluna_dois")
        )
df.explain()